<div style="
    width: 100%;
    box-sizing: border-box;
    margin: 20px 0;
    padding: 20px;
    background: #0b0f1a;
    border: 1px solid rgba(0,255,255,0.2);
    border-radius: 12px;
    color: #d6faff;
    font-family: Arial, sans-serif;
">

<h2 style="color:#00f5ff;">⚡ The Agent Loop — LLM Tool Use & Autonomous Planning Exercise</h2>

<p>
Build a fully autonomous <strong>Agent Loop</strong> from scratch using an LLM and a set of callable tools.
The agent must solve a multi-step problem by first using <span style="color:#00f5ff;">create_todos</span> to plan a structured list of steps,
then calling <span style="color:#00f5ff;">mark_complete</span> to tick off each step as it executes — producing a visible, traceable reasoning trail rendered via <span style="color:#39ff14;">Rich console markup</span>.
</p>

<p>
Define tool schemas in <strong>JSON format</strong> and register them with the LLM via the <code>tools</code> parameter.
The agent must autonomously decide when to call each tool, what arguments to pass, and how to interpret the results —
managing a live <span style="color:#00f5ff;">todo list</span> and a <span style="color:#00f5ff;">completed list</span> in memory across multiple LLM turns.
</p>

<p>
Implement the full <strong>agentic loop</strong> — if the model returns <code>finish_reason == "tool_calls"</code>,
execute the requested tool via <code>globals().get(tool_name)</code>, append the result back into the message history,
and re-invoke the LLM automatically until it returns a final text response, completing the cycle.
</p>

<h2 style="color:#ff7800; margin-top: 20px;">🚀 Exercise — Build It From Scratch</h2>

<p>Now construct your own Agent Loop independently in a new <code>.ipynb</code> file, from first principles:</p>

<ol style="line-height: 2;">
  <li>Define two tools — <span style="color:#00f5ff;">create_todos</span> (accepts a list of step descriptions) and <span style="color:#00f5ff;">mark_complete</span> (accepts an index and completion notes) — and write their JSON schemas.</li>
  <li>Write a <code>handle_tool_calls()</code> function that dispatches tool calls returned by the LLM and formats results as <code>role: "tool"</code> messages.</li>
  <li>Write a <code>loop(messages)</code> function that keeps calling the LLM, handling tool calls until <code>finish_reason != "tool_calls"</code>.</li>
  <li>Write a system prompt instructing the LLM to plan using the todo tools before solving any problem, then provide a real word problem as the user message.</li>
  <li>Run the loop and observe the agent autonomously plan, execute, and mark off steps — arriving at a final answer without any human intervention.</li>
</ol>

</div>

In [ ]:
# Import relevant libraries

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os

load_dotenv(override=True)  # Load environment variables from .env file, overriding existing ones if necessary

True

In [ ]:
# Get the OpenAI API key from environment variables
openai_api_key = os.getenv('NVIDIA_API_KEY_1')

In [ ]:
# Console printing function with fallback to standard print from rich library
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [ ]:
# Initialize the OpenAI client with the API key and base URL
openai = OpenAI(
    base_url = "https://integrate.api.nvidia.com/v1", 
    api_key = openai_api_key
)

In [ ]:
# Create empty lists for todos and completed tasks

todos = []
completed = []

In [ ]:
# Function to add a todo item to the list
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [ ]:
get_todo_report()

''

In [ ]:
# Function to add multiple todo items to the list
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [ ]:
# Function to mark a todo item as complete based on its index
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [ ]:
todos, completed = [], []

# Create initial todo items
create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [ ]:
mark_complete(1, "bought")  # Mark the first todo as complete with a note

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [ ]:
# Define the JSON schema for the create_todos function to be used in the agent's tool specification
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [ ]:
# Define the JSON schema for the mark_complete function to be used in the agent's tool specification
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [ ]:
# Define the list of tools for the agent, including the JSON schemas for each function
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [ ]:
# Function to handle tool calls made by the agent, executing the corresponding functions and returning results
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
# Main loop to interact with the agent, processing its responses and handling tool calls until a final response is generated
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages, tools=tools, reasoning_effort="low")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [ ]:
# Define the system message and user message for the agent's initial prompt
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""

# Combine the system message and user message into a list of messages to be sent to the agent
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [ ]:
todos, completed = [], []
loop(messages)  # Start the interaction loop with the agent using the defined messages

Todo #1: Determine the distance between Boston and New York (assume 215 miles).
Todo #2: Calculate the distance the Boston train travels before the NY train starts.
Todo #3: Set up equation for meeting time after 3:00 pm.
Todo #4: Solve for meeting time.
Todo #5: Provide answer in HH:MM format.

Assumed distance Boston‑NY ≈ 215 miles (typical highway distance).

Todo #1: Determine the distance between Boston and New York (assume 215 miles).
Todo #2: Calculate the distance the Boston train travels before the NY train starts.
Todo #3: Set up equation for meeting time after 3:00 pm.
Todo #4: Solve for meeting time.
Todo #5: Provide answer in HH:MM format.

Boston train travels 60 miles in first hour.

Todo #1: Determine the distance between Boston and New York (assume 215 miles).
Todo #2: Calculate the distance the Boston train travels before the NY train starts.
Todo #3: Set up equation for meeting time after 3:00 pm.
Todo #4: Solve for meeting time.
Todo #5: Provide answer in HH:MM format.

Remaining distance 155 miles, closing speed 140 mph, time = 155/140 hr.

Todo #1: Determine the distance between Boston and New York (assume 215 miles).
Todo #2: Calculate the distance the Boston train travels before the NY train starts.
Todo #3: Set up equation for meeting time after 3:00 pm.
Todo #4: Solve for meeting time.
Todo #5: Provide answer in HH:MM format.

Calculated meeting time = 3:00 + 1.107 hr ≈ 4:06 pm.

Todo #1: Determine the distance between Boston and New York (assume 215 miles).
Todo #2: Calculate the distance the Boston train travels before the NY train starts.
Todo #3: Set up equation for meeting time after 3:00 pm.
Todo #4: Solve for meeting time.
Todo #5: Provide answer in HH:MM format.

**Solution**

- Distance Boston ↔ New York (assumed) = **215 mi**  
- Boston train travels for 1 hour before the NY train starts → **60 mi** covered.  
- Remaining distance = 215 mi − 60 mi = **155 mi**  
- Closing speed = 60 mph + 80 mph = **140 mph**  
- Time to meet after 3:00 pm = 155 mi ÷ 140 mph ≈ **1.107 h** ≈ **1 h 6 min**  

**Meeting time:** 3:00 pm + 1 h 6 min ≈ **4:06 pm**.